# Topic Modeling — LDA, LSA, and NMF

This notebook applies three topic modeling techniques on a text document:
- **LSA** (Latent Semantic Analysis): uses SVD on TF-IDF matrix
- **LDA** (Latent Dirichlet Allocation): probabilistic model using word counts
- **NMF** (Non-negative Matrix Factorization): factorizes TF-IDF into non-negative parts

In [ ]:
# All required libraries are part of scikit-learn — no extra install needed
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation, NMF

## Step 1: The Text Document

We use a long essay about Uttarakhand. The document is split into paragraphs, and each paragraph is treated as a separate 'document' for topic modeling.

In [ ]:
# A detailed essay about Uttarakhand covering geography, culture, religion, and nature
doc = '''Uttarakhand: Geography, Culture, Religion, and Natural Wealth

Uttarakhand is a northern state of India located in the lap of the Himalayas. It was formed in the year 2000 after being carved out of Uttar Pradesh. The state is bordered by Tibet to the north, Nepal to the east, Himachal Pradesh to the west, and Uttar Pradesh to the south. Due to its strategic location and unique topography, Uttarakhand holds great geographical, cultural, and environmental importance. The state is divided into two main regions, Garhwal and Kumaon, both of which have distinct identities yet share common traditions and lifestyles.

The geography of Uttarakhand is dominated by mountainous terrain, deep valleys, glaciers, and river systems. The Himalayas form the backbone of the state, with snow-covered peaks and rugged landscapes shaping its natural environment. Some of the major rivers of India, including the Ganga and Yamuna, originate from glaciers located in Uttarakhand. The Gangotri Glacier is the source of the Ganga, while the Yamunotri Glacier gives rise to the Yamuna River. These rivers are not only crucial for the ecology of northern India but also hold immense religious significance.

The climate of Uttarakhand varies significantly depending on altitude. The lower regions experience a subtropical climate with hot summers and mild winters, while the higher altitudes have cold weather with heavy snowfall during winters. This variation in climate supports a wide range of vegetation and biodiversity. Forests cover a large portion of the state and include species such as pine, oak, deodar, and rhododendron. These forests play a vital role in maintaining ecological balance and supporting wildlife.

Tourism is one of the most important sectors of Uttarakhand economy. The state is widely known for its scenic beauty, attracting tourists from across India and around the world. Hill stations such as Nainital, Mussoorie, and Ranikhet are popular destinations known for their pleasant climate and picturesque views. These places offer a peaceful retreat from the crowded urban environment and are especially popular during the summer months.

Adventure tourism has also gained popularity in Uttarakhand. Activities such as trekking, river rafting, skiing, and camping attract thrill-seekers and nature enthusiasts. The mountainous terrain provides ideal conditions for trekking routes, including famous trails like the Valley of Flowers trek and the Roopkund trek. River rafting in the Ganga, particularly in Rishikesh, is a major attraction for adventure lovers.

Religious tourism plays a central role in Uttarakhand identity. The state is often referred to as Devbhumi, meaning the Land of Gods. It is home to several important Hindu pilgrimage sites that attract millions of devotees each year. The Char Dham Yatra, consisting of Kedarnath, Badrinath, Gangotri, and Yamunotri, is one of the most significant religious journeys in India. These sites are located in the higher Himalayan regions and are visited by pilgrims seeking spiritual fulfillment.

Kedarnath is dedicated to Lord Shiva and is one of the twelve Jyotirlingas. Badrinath, dedicated to Lord Vishnu, is another major pilgrimage site. Gangotri and Yamunotri are associated with the origins of the sacred rivers Ganga and Yamuna. Apart from the Char Dham, Uttarakhand is also home to other important religious centers such as Haridwar and Rishikesh. Haridwar is known for the Ganga Aarti performed on the banks of the river, while Rishikesh is considered the yoga capital of the world.

The cultural heritage of Uttarakhand is rich and diverse, reflecting the traditions of its people. The population mainly consists of Garhwali and Kumaoni communities, each with its own language, customs, and practices. Festivals are an integral part of life in Uttarakhand and are celebrated with great enthusiasm. Major festivals include Makar Sankranti, Holi, Diwali, and local fairs such as Nanda Devi Raj Jat Yatra.

Agriculture is a primary occupation for a large portion of the population. Due to the hilly terrain, terrace farming is commonly practiced. Crops such as rice, wheat, millets, and pulses are grown in different regions. Horticulture is also significant, with fruits like apples, pears, and plums being cultivated in higher altitudes. Despite these activities, agriculture faces challenges due to limited land availability and dependence on rainfall.

Uttarakhand is also known for its rich biodiversity. The state has several national parks and wildlife sanctuaries that protect its flora and fauna. Jim Corbett National Park, established in 1936, is the oldest national park in India and is famous for its population of Bengal tigers. Other protected areas include Rajaji National Park and the Valley of Flowers National Park, which is a UNESCO World Heritage Site.

Environmental conservation is a critical issue in Uttarakhand. The state is prone to natural disasters such as landslides, floods, and earthquakes due to its fragile mountainous ecosystem. Climate change has further intensified these risks, affecting glaciers and water resources. Sustainable development and responsible tourism are necessary to preserve the natural beauty and ecological balance of the region.

The economy of Uttarakhand is based on a combination of agriculture, tourism, and small-scale industries. Handicrafts and traditional products such as woolen garments and wooden items are produced in rural areas. Industrial development has also taken place in certain regions, particularly in areas like Haridwar and Dehradun.

Education and infrastructure in Uttarakhand have improved over the years, but challenges remain in remote and mountainous regions. Access to healthcare, transportation, and education is still limited in many areas. Efforts are being made by the government to improve connectivity and provide better facilities to the population.

Migration is another issue faced by the state. Many people from rural areas move to cities in search of better employment opportunities. This has led to depopulation in some villages, affecting local economies and traditional lifestyles. Addressing this issue requires creating sustainable livelihoods within the state.
-----------'''

# Split document into paragraphs — each paragraph = one document for modeling
docs = [p.strip() for p in doc.split('\n') if len(p.strip()) > 30]

print(f"Total paragraphs (documents): {len(docs)}")
print(f"\nSample paragraph:")
print(docs[1][:200], "...")

Total paragraphs (documents): 15

Sample paragraph:
Uttarakhand is a northern state of India located in the lap of the Himalayas. It was formed in the year 2000 after being carved out of Uttar Pradesh. The state is bordered by Tibet to the north, Nepal ...


## Step 2: Vectorize the Text

We need to convert text into numbers before applying any topic model.
- **TF-IDF** is used for LSA and NMF — it captures word importance across documents.
- **Count Vectorizer** is used for LDA — LDA is a probabilistic model that needs raw word counts, not weighted scores.

In [ ]:
# TF-IDF Vectorizer: gives higher weight to rare but meaningful words
# stop_words='english' removes common words like 'the', 'is', 'and'
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(docs)

# Count Vectorizer: simply counts how many times each word appears
# LDA works with raw counts, not TF-IDF weights
count_vectorizer = CountVectorizer(stop_words='english')
X_count = count_vectorizer.fit_transform(docs)

print(f"TF-IDF matrix shape : {X_tfidf.shape}  (docs x unique_words)")
print(f"Count matrix shape  : {X_count.shape}  (docs x unique_words)")

TF-IDF matrix shape : (15, 374)  (docs x unique_words)
Count matrix shape  : (15, 374)  (docs x unique_words)


## Step 3: LSA (Latent Semantic Analysis)

LSA uses **Truncated SVD** (Singular Value Decomposition) on the TF-IDF matrix. It reduces the high-dimensional word space into a lower-dimensional topic space.

In [ ]:
from sklearn.decomposition import TruncatedSVD

# n_components = number of topics we want to discover
lsa = TruncatedSVD(n_components=3, random_state=42)
lsa.fit(X_tfidf)

# Get the vocabulary (all unique words after vectorization)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print("LSA Topics (top 10 words per topic):\n")
for i, component in enumerate(lsa.components_):
    # argsort() returns indices from lowest to highest
    # [-10:] picks the 10 highest scoring words for this topic
    top_word_indices = component.argsort()[-10:]
    top_words = [tfidf_terms[j] for j in top_word_indices]
    print(f"Topic {i+1}: {top_words}")

LSA Topics (top 10 words per topic):

Topic 1: ['glaciers', 'rivers', 'yamuna', 'india', 'religious', 'natural', 'river', 'ganga', 'state', 'uttarakhand']
Topic 2: ['education', 'traditional', 'rural', 'better', 'sustainable', 'affecting', 'state', 'climate', 'areas', 'issue']
Topic 3: ['state', 'environmental', 'beauty', 'issue', 'glaciers', 'wealth', 'religion', 'culture', 'geography', 'natural']


## Step 4: LDA (Latent Dirichlet Allocation)

LDA is a generative probabilistic model. It assumes each document is a mixture of topics, and each topic is a distribution over words. It uses word counts (not TF-IDF).

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# n_components = number of topics
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(X_count)

# Get the vocabulary for the count vectorizer
count_terms = count_vectorizer.get_feature_names_out()

print("LDA Topics (top 10 words per topic):\n")
for i, component in enumerate(lda.components_):
    # Higher value = stronger association between word and that topic
    top_word_indices = component.argsort()[-10:]
    top_words = [count_terms[j] for j in top_word_indices]
    print(f"Topic {i+1}: {top_words}")

LDA Topics (top 10 words per topic):

Topic 1: ['economy', 'lifestyles', 'popular', 'uttar', 'known', 'issue', 'india', 'pradesh', 'uttarakhand', 'state']
Topic 2: ['river', 'india', 'yamuna', 'rivers', 'terrain', 'agriculture', 'glaciers', 'mountainous', 'natural', 'uttarakhand']
Topic 3: ['rishikesh', 'include', 'major', 'religious', 'tourism', 'climate', 'areas', 'park', 'national', 'uttarakhand']


## Step 5: NMF (Non-negative Matrix Factorization)

NMF factorizes the TF-IDF matrix into two non-negative matrices: one for document-topic weights, one for topic-word weights. Non-negativity makes topics easier to interpret.

In [ ]:
from sklearn.decomposition import NMF

# NMF also uses TF-IDF (like LSA), not raw counts
nmf = NMF(n_components=3, random_state=42)
nmf.fit(X_tfidf)

print("NMF Topics (top 10 words per topic):\n")
for i, component in enumerate(nmf.components_):
    top_word_indices = component.argsort()[-10:]
    top_words = [tfidf_terms[j] for j in top_word_indices]
    print(f"Topic {i+1}: {top_words}")

NMF Topics (top 10 words per topic):

Topic 1: ['gangotri', 'yamunotri', 'dedicated', 'lord', 'rishikesh', 'rivers', 'yamuna', 'religious', 'river', 'ganga']
Topic 2: ['wealth', 'culture', 'religion', 'affecting', 'sustainable', 'uttarakhand', 'climate', 'issue', 'state', 'natural']
Topic 3: ['access', 'connectivity', 'population', 'like', 'challenges', 'limited', 'regions', 'education', 'areas', 'agriculture']


## Conclusion

| Model | Input | Method | Best For |
|-------|-------|--------|----------|
| **LSA** | TF-IDF | SVD (linear algebra) | Finding latent dimensions in large corpora |
| **LDA** | Word Counts | Probabilistic (EM) | Short documents, interpretable topics |
| **NMF** | TF-IDF | Matrix factorization | Sparse data, clean topic separation |

- All three models found topics related to **religion/pilgrimage**, **geography/environment**, and **tourism/infrastructure** — which matches the document structure.
- NMF usually gives the most interpretable topics because of the non-negativity constraint.